# ML Ops project
*Détecter les caractéristiques d'un visage*

DEFIVES François

GRANIER Sylvain

HACALA Maude

### 0. Vérification des prérequis

In [5]:
import sys
if '3.13.7' not in sys.version:
    print("Veuillez utiliser la version 3.13.7 de Python")
import pandas as pd
import torch
# import pyspark
import time
import PIL
from PIL import Image
import numpy as np
import os
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import udf, col
# from pyspark.sql.types import BinaryType, StringType
from PIL import Image
import numpy as np
import io
import os

### Import des données annotés 1

Chaque image annottée dans le dossier /data/lot1 est composée d'un fichier csv associé contenant ses caractéristiques.

### Description des données

### Traitement des images

Nous souhaitons redimensionner toutes les images à une taille de 64*64 et segmenter les visages et centrer sur le visage.

In [6]:
# Segmentation simple + redimensionnement des images du dossier
time_start=time.time()
input_folder = 'data/test_débile/'
output_folder = 'data/test_débile_resized/'
target_size = (64, 64)

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith('.png'):
        continue

    img_path = os.path.join(input_folder, filename)
    img = Image.open(img_path).convert("RGB")

    # Passage en niveaux de gris pour simplifier
    gray = img.convert("L")
    arr = np.array(gray)

    # Masque : pixels non blancs (ou pas trop blancs)
    # Tu peux ajuster le seuil 250 (0 = noir, 255 = blanc)
    mask = arr < 250

    # Si tout est blanc (image vide), on évite de planter
    if not mask.any():
        cropped = img
    else:
        # Coordonnées des pixels “utiles”
        ys, xs = np.where(mask)
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        # Recadrage sur la zone utile
        cropped = img.crop((x_min, y_min, x_max + 1, y_max + 1))

    # Redimensionnement
    img_resized = cropped.resize(target_size)
    img_resized.save(os.path.join(output_folder, filename))
time_end=time.time()
print(f"Temps de traitement des images : {time_end - time_start} secondes")

Temps de traitement des images : 0.09388279914855957 secondes


### Traitement des .csv

In [7]:
time_start=time.time()
input_folder = 'data/lot1_images/'
output_folder = 'data/lot1_resized/'

beard = pd.read_csv("beard.csv", sep=";") 
mustache = pd.read_csv("mustache.csv", sep=";") 
glasses = pd.read_csv("glasses.csv", sep=";") 
hair_color = pd.read_csv("hair_color.csv", sep=";") 
hair = pd.read_csv("hair.csv", sep=";")

# boucle sur les fichiers csv dans data/lot1/
for file in os.listdir(input_folder):
    if file.endswith(".csv"):

        df = pd.read_csv(os.path.join(input_folder, file), header=None)
        df[0] = df[0].str.replace('"', '').str.strip()
        
        row = df[df[0] == 'glasses']
        value = row.iloc[0, 1]
        if(value in glasses['yes'].values):
            glasses_value = 1
        else:
            glasses_value = 0

        row = df[df[0] == 'facial_hair']
        value = row.iloc[0, 1]
        if(value in beard['yes'].values):
            beard_value = 1
        else:
            beard_value = 0
        
        if(value in mustache['yes'].values):
            mustache_value = 1
        else:
            mustache_value = 0

        row = df[df[0] == 'hair_color']
        value = row.iloc[0, 1]
        if(value in hair_color['blond'].values):
            blond_value = 1   
        else:
            blond_value = 0
        
        if(value in hair_color['light_brown'].values):
            light_brown_value = 1
        else:
            light_brown_value = 0     

        if(value in hair_color['dark_brown'].values):
            dark_brown_value = 1  
        else:
            dark_brown_value = 0

        if(value in hair_color['redhead'].values):
            redhead_value = 1  
        else:
            redhead_value = 0

        if(value in hair_color['gray_blue'].values):
            gray_blue_value= 1
        else:
            gray_blue_value = 0

        row = df[df[0] == 'hair']
        value = row.iloc[0, 1]
        if(value in hair['long'].values):
            long_value = 1
        else:
            long_value = 0

        if(value in hair['short'].values):
            short_value = 1
        else:
            short_value = 0

        if(value in hair['bald'].values):
            bald_value = 1
        else:
            bald_value = 0

        #créer nouveau fichier csv avec les valeurs extraites
        output_file = os.path.join(os.path.join(output_folder, file))
        with open(output_file, 'w') as f:
            f.write(f"glasses;{glasses_value}\n")
            f.write(f"beard;{beard_value}\n")
            f.write(f"mustache;{mustache_value}\n")
            f.write(f"blond;{blond_value}\n")
            f.write(f"light_brown;{light_brown_value}\n")
            f.write(f"dark_brown;{dark_brown_value}\n")
            f.write(f"redhead;{redhead_value}\n")
            f.write(f"gray_blue;{gray_blue_value}\n")
            f.write(f"long;{long_value}\n")
            f.write(f"short;{short_value}\n")
            f.write(f"bald;{bald_value}\n")

time_end=time.time()
print(f"Temps de traitement des .csv : {time_end - time_start} secondes")

KeyboardInterrupt: 

### Spark

### EN TRAIN DE MOURIR A CAUSE DE CE CONNARDS DE SPARK

### CNN

In [8]:
import os
import time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# ============================================================
# 1. CONFIG
# ============================================================

# À adapter à ta structure de dossiers
IMAGE_DIR = "data/lot1_resized"   # dossier qui contient les images (.jpg, .png, ...)
LABEL_DIR = "data/lot1_resized"         # dossier qui contient les CSV générés (glasses;..., etc.)

BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
VAL_RATIO = 0.2
RANDOM_SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

# Transformations sur les images
transform = transforms.Compose([
    transforms.ToTensor(),
    # Normalisation type ImageNet (à adapter si besoin)
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 2. DATASET
# ============================================================

class FaceDataset(Dataset):
    """
    Dataset qui lit :
      - une image dans IMAGE_DIR (jpg/png)
      - un CSV correspondant dans LABEL_DIR, au format :
            glasses;0/1
            beard;0/1
            mustache;0/1
            blond;0/1
            light_brown;0/1
            dark_brown;0/1
            redhead;0/1
            gray_blue;0/1
            long;0/1
            short;0/1
            bald;0/1
    Et renvoie :
      - image : tensor (C,H,W)
      - glasses, beard, mustache : float tensor (0./1.)
      - color : long tensor dans [0..4]
      - hair  : long tensor dans [0..2]
    """

    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform

        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if len(self.images) == 0:
            raise RuntimeError(f"Aucune image trouvée dans {image_dir}")

        # Ordre des classes pour color et hair
        self.color_attrs = ["blond", "light_brown", "dark_brown", "redhead", "gray_blue"]
        self.hair_attrs  = ["long", "short", "bald"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        base = os.path.splitext(img_name)[0]

        img_path = os.path.join(self.image_dir, img_name)
        csv_path = os.path.join(self.label_dir, base + ".csv")

        # --- Image ---
        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        # --- Labels ---
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV labels manquant pour {img_name} : {csv_path}")

        df = pd.read_csv(csv_path, sep=";", header=None, names=["attr", "val"])
        labels = dict(zip(df["attr"], df["val"]))

        def get_bin(attr):
            # renvoie 0.0 si inexistant
            return torch.tensor(float(labels.get(attr, 0)), dtype=torch.float32)

        def get_multiclass_from_onehot(attr_list):
            # ex : [blond, light_brown, dark_brown, redhead, gray_blue]
            vals = [int(labels.get(a, 0)) for a in attr_list]
            return torch.tensor(int(np.argmax(vals)), dtype=torch.long)

        # Binaires
        y_glasses  = get_bin("glasses")
        y_beard    = get_bin("beard")
        y_mustache = get_bin("mustache")

        # Multi-classes
        y_color = get_multiclass_from_onehot(self.color_attrs)  # 0..4
        y_hair  = get_multiclass_from_onehot(self.hair_attrs)   # 0..2

        sample = {
            "image": image,
            "glasses": y_glasses,
            "beard": y_beard,
            "mustache": y_mustache,
            "color": y_color,
            "hair": y_hair,
        }

        return sample

# ============================================================
# 3. MODELE CNN MULTI-TÂCHES
#    (si tu as déjà un CNNMultiTask, utilise le tien)
# ============================================================

class CNNMultiTask(nn.Module):
    def __init__(self, input_size=(3, 64, 64)):
        super().__init__()

        # Image de base 64*64
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # -> (32, 128, 128)
            nn.ReLU(),
            nn.MaxPool2d(2),                             # -> (32, 64, 64)

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # -> (64, 64, 64)
            nn.ReLU(),
            nn.MaxPool2d(2),                             # -> (64, 32, 32)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),# -> (128, 32, 32)
            nn.ReLU(),
            nn.MaxPool2d(2),                             # -> (128, 16, 16)
        )

         # calcule dynamique de la dimension aplatie en passant un tenseur dummy
        with torch.no_grad():
            dummy = torch.zeros(1, *input_size)
            feat = self.features(dummy)
            self.flatten_dim = feat.view(1, -1).size(1)

        # fully connected partagé
        self.shared_fc = nn.Sequential(
            nn.Linear(self.flatten_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Têtes pour chaque tâche
        self.fc_glasses  = nn.Linear(256, 1)  # logit binaire
        self.fc_beard    = nn.Linear(256, 1)
        self.fc_mustache = nn.Linear(256, 1)

        self.fc_color = nn.Linear(256, 5)     # 5 classes pour la couleur
        self.fc_hair  = nn.Linear(256, 3)     # 3 classes pour la coupe

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        glasses_logits  = self.fc_glasses(x).squeeze(1)   # (B,)
        beard_logits    = self.fc_beard(x).squeeze(1)     # (B,)
        must_logits     = self.fc_mustache(x).squeeze(1)  # (B,)
        color_logits    = self.fc_color(x)                # (B,5)
        hair_logits     = self.fc_hair(x)                 # (B,3)

        return glasses_logits, beard_logits, must_logits, color_logits, hair_logits

# ============================================================
# 4. DATA LOADERS (train / val split)
# ============================================================

full_dataset = FaceDataset(IMAGE_DIR, LABEL_DIR, transform=transform)

torch.manual_seed(RANDOM_SEED)
train_size = int((1 - VAL_RATIO) * len(full_dataset))
val_size   = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size : {len(train_dataset)}")
print(f"Val size   : {len(val_dataset)}")

# ============================================================
# 5. INITIALISATION MODELE + LOSSES + OPTIM
# ============================================================

model = CNNMultiTask().to(device)

BCE = nn.BCEWithLogitsLoss()
CE  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ============================================================
# 6. BOUCLE D'ENTRAÎNEMENT + VALIDATION
# ============================================================

for epoch in range(1, EPOCHS + 1):
    # ------------------
    #       TRAIN
    # ------------------
    model.train()
    running_loss = 0.0

    train_acc_glasses = 0.0
    train_acc_beard   = 0.0
    train_acc_must    = 0.0
    train_acc_color   = 0.0
    train_acc_hair    = 0.0

    for batch in train_loader:
        images      = batch["image"].to(device)
        y_glasses   = batch["glasses"].to(device)
        y_beard     = batch["beard"].to(device)
        y_mustache  = batch["mustache"].to(device)
        y_color     = batch["color"].to(device)
        y_hair      = batch["hair"].to(device)

        optimizer.zero_grad()

        glasses_logits, beard_logits, must_logits, color_logits, hair_logits = model(images)

        # Loss
        loss_glasses = BCE(glasses_logits, y_glasses)
        loss_beard   = BCE(beard_logits,   y_beard)
        loss_must    = BCE(must_logits,    y_mustache)
        loss_color   = CE(color_logits,    y_color)
        loss_hair    = CE(hair_logits,     y_hair)

        loss = loss_glasses + loss_beard + loss_must + loss_color + loss_hair

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Accuracies binaires
        pred_glasses = (torch.sigmoid(glasses_logits) > 0.5).float()
        pred_beard   = (torch.sigmoid(beard_logits)   > 0.5).float()
        pred_must    = (torch.sigmoid(must_logits)    > 0.5).float()

        train_acc_glasses += (pred_glasses == y_glasses).float().mean().item()
        train_acc_beard   += (pred_beard   == y_beard).float().mean().item()
        train_acc_must    += (pred_must    == y_mustache).float().mean().item()

        # Accuracies multi-class
        pred_color = torch.argmax(color_logits, dim=1)
        pred_hair  = torch.argmax(hair_logits,  dim=1)

        train_acc_color += (pred_color == y_color).float().mean().item()
        train_acc_hair  += (pred_hair  == y_hair).float().mean().item()

    train_loss = running_loss / len(train_loader)
    train_acc_glasses /= len(train_loader)
    train_acc_beard   /= len(train_loader)
    train_acc_must    /= len(train_loader)
    train_acc_color   /= len(train_loader)
    train_acc_hair    /= len(train_loader)

    # ------------------
    #       VALIDATION
    # ------------------
    model.eval()
    val_loss = 0.0

    val_acc_glasses = 0.0
    val_acc_beard   = 0.0
    val_acc_must    = 0.0
    val_acc_color   = 0.0
    val_acc_hair    = 0.0

    with torch.no_grad():
        for batch in val_loader:
            images      = batch["image"].to(device)
            y_glasses   = batch["glasses"].to(device)
            y_beard     = batch["beard"].to(device)
            y_mustache  = batch["mustache"].to(device)
            y_color     = batch["color"].to(device)
            y_hair      = batch["hair"].to(device)

            glasses_logits, beard_logits, must_logits, color_logits, hair_logits = model(images)

            loss_glasses = BCE(glasses_logits, y_glasses)
            loss_beard   = BCE(beard_logits,   y_beard)
            loss_must    = BCE(must_logits,    y_mustache)
            loss_color   = CE(color_logits,    y_color)
            loss_hair    = CE(hair_logits,     y_hair)

            loss = loss_glasses + loss_beard + loss_must + loss_color + loss_hair
            val_loss += loss.item()

            # Accuracies binaires
            pred_glasses = (torch.sigmoid(glasses_logits) > 0.5).float()
            pred_beard   = (torch.sigmoid(beard_logits)   > 0.5).float()
            pred_must    = (torch.sigmoid(must_logits)    > 0.5).float()

            val_acc_glasses += (pred_glasses == y_glasses).float().mean().item()
            val_acc_beard   += (pred_beard   == y_beard).float().mean().item()
            val_acc_must    += (pred_must    == y_mustache).float().mean().item()

            # Accuracies multi-class
            pred_color = torch.argmax(color_logits, dim=1)
            pred_hair  = torch.argmax(hair_logits,  dim=1)

            val_acc_color += (pred_color == y_color).float().mean().item()
            val_acc_hair  += (pred_hair  == y_hair).float().mean().item()

    val_loss /= len(val_loader)
    val_acc_glasses /= len(val_loader)
    val_acc_beard   /= len(val_loader)
    val_acc_must    /= len(val_loader)
    val_acc_color   /= len(val_loader)
    val_acc_hair    /= len(val_loader)

    print(f"[Epoch {epoch}/{EPOCHS}] "
          f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
          f"G(acc) T={train_acc_glasses:.3f} V={val_acc_glasses:.3f} | "
          f"B(acc) T={train_acc_beard:.3f} V={val_acc_beard:.3f} | "
          f"M(acc) T={train_acc_must:.3f} V={val_acc_must:.3f} | "
          f"C(acc) T={train_acc_color:.3f} V={val_acc_color:.3f} | "
          f"H(acc) T={train_acc_hair:.3f} V={val_acc_hair:.3f}"
    )


Device : cuda
Train size : 12712
Val size   : 3178


KeyboardInterrupt: 

In [10]:
## Sauvegarde du modèle entraîné
model_path = "cnn_multitask_model.pth"
# torch.save(model.state_dict(), model_path)

## Prédiction sur un jeu de données

In [11]:
# Chargement du modèle sauvegardé
model = CNNMultiTask().to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

IMAGE_DIR = "data/test_débile_resized"

# Chargement du jeu de données pour prédiction sans CSV de labels
class FaceDatasetNoLabels(Dataset):
    """
    Dataset qui lit uniquement les images dans IMAGE_DIR (jpg/png)
    et renvoie :
      - image : tensor (C,H,W)
      - filename : nom du fichier image
    """

    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform

        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if len(self.images) == 0:
            raise RuntimeError(f"Aucune image trouvée dans {image_dir}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)

        # --- Image ---
        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        sample = {
            "image": image,
            "filename": img_name,
        }

        return sample

# Chargement du jeu de données pour prédiction
test_dataset = FaceDatasetNoLabels(IMAGE_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Préparation du fichier de sortie
output_csv = "predictions.csv"
with open(output_csv, "w") as f:
    f.write("filename,glasses,beard,mustache,blond,light_brown,dark_brown,redhead,gray_blue,long,short,bald\n")

# Prédictions
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        filenames = [test_dataset.images[idx] for idx in range(len(batch["image"]))]

        glasses_logits, beard_logits, must_logits, color_logits, hair_logits = model(images)

        # Calcul des prédictions
        pred_glasses = (torch.sigmoid(glasses_logits) > 0.5).int().cpu().numpy()
        pred_beard = (torch.sigmoid(beard_logits) > 0.5).int().cpu().numpy()
        pred_mustache = (torch.sigmoid(must_logits) > 0.5).int().cpu().numpy()
        pred_color = torch.argmax(color_logits, dim=1).cpu().numpy()
        pred_hair = torch.argmax(hair_logits, dim=1).cpu().numpy()

        # Conversion des prédictions multi-classes en one-hot
        pred_color_onehot = np.eye(5)[pred_color]
        pred_hair_onehot = np.eye(3)[pred_hair]

        # Écriture dans le fichier CSV
        with open(output_csv, "a") as f:
            for i, filename in enumerate(filenames):
                row = [filename]
                row.append(pred_glasses[i])
                row.append(pred_beard[i])
                row.append(pred_mustache[i])
                row.extend(pred_color_onehot[i])
                row.extend(pred_hair_onehot[i])
                f.write(",".join(map(str, row)) + "\n")



print(f"Prédictions sauvegardées dans {output_csv}")


Prédictions sauvegardées dans predictions.csv


# Mise en forme de la prédiciton 

In [ ]:
# Ouverture du fichier des prédictions et affichage
import pandas as pd
predictions_df = pd.read_csv("predictions.csv")
predictions_df.head()

# Correspondance des valeurs pour taille_cheveux et couleur_cheveux
taille_cheveux_mapping = {0: "chauve", 1: "court", 2: "long"}
couleur_cheveux_mapping = {0: "blond", 1: "chatain", 2: "roux", 3: "brun", 4: "gris/bleu"}

# Ajout des colonnes correspondantes
predictions_df["taille_cheveux"] = predictions_df[["bald", "short", "long"]].idxmax(axis=1).map({"bald": 0, "short": 1, "long": 2})
predictions_df["couleur_cheveux"] = predictions_df[["blond", "light_brown", "redhead", "dark_brown", "gray_blue"]].idxmax(axis=1).map({"blond": 0, "light_brown": 1, "redhead": 2, "dark_brown": 3, "gray_blue": 4})

# Mise à jour du vecteur final avec les nouvelles colonnes
final_df = predictions_df.rename(columns={
    "filename": "image_name",
    "beard": "barbe",
    "mustache": "moustache",
    "glasses": "lunettes"
})[["image_name", "barbe", "moustache", "lunettes", "taille_cheveux", "couleur_cheveux"]]

# Affichage du vecteur final
print(final_df)

# Sauvegarde du vecteur final dans un nouveau fichier CSV
final_df.to_csv("final_predictions.csv", index=False)

                              image_name  barbe  moustache  lunettes  \
0  20250828_1306521-removebg-preview.png      1          0         1   
1                            debile1.png      0          0         1   
2                            debile2.png      0          0         1   

   taille_cheveux  couleur_cheveux  
0               2                3  
1               2                4  
2               2                3  
